# VECTOR RUN: real GPU benchmark
専用 GPU runtime を選択し、ゲーム最新版・監査済み manifest・server・experiments を `/content/vector-run` に配置する。公開 repo を利用する場合は親が統合した commit を固定する。
モデルの system_one のみを使い、60 fps の realtime physics を維持する。3 問 seeds 1,2,3 と 64 問 seed 1 は別系列。mock/ルール fallback は使わない。異なる GPU / dtype / quantization は別条件。
成果物 download 後、使用した専用 Colab session のみ停止する。既存の他 session を変更しない。公開前に notebook outputs を消す。


In [ ]:
# 親が統合した公開 commit の SHA に置換する。
GAME_COMMIT = 'REPLACE_WITH_PARENT_INTEGRATED_COMMIT_SHA'
MODEL = 'kai'  # eos, sol, sol-reasoning, nox, lux, vega
import pathlib, subprocess
root = pathlib.Path('/content/vector-run')
assert len(GAME_COMMIT) == 40 and all(c in '0123456789abcdef' for c in GAME_COMMIT.lower())
if not root.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Sunwood-ai-labs/vector-run-decision-colab.git', str(root)], check=True)
subprocess.run(['git', 'fetch', 'origin', GAME_COMMIT], cwd=root, check=True)
subprocess.run(['git', 'checkout', '--detach', GAME_COMMIT], cwd=root, check=True)


In [ ]:
subprocess.run(['bash', 'experiments/colab/bootstrap.sh'], cwd=root, check=True)


In [ ]:
import os
uv = str(pathlib.Path.home() / '.local/bin/uv')
if not pathlib.Path(uv).exists():
    uv = 'uv'
command = [uv, 'run', '--no-project', '--no-managed-python', '--no-sync', 'python', 'experiments/colab/run_bench.py', '--models', MODEL, '--output-dir', f'results/{MODEL}', '--game-commit', GAME_COMMIT]
completed = subprocess.run(command, cwd=root)
print({'runner_exit': completed.returncode, 'result_files': [p.name for p in (root / 'results' / MODEL).glob('*.json')]})
# nonzero も OOM/blocked/失敗 JSON として保存する。成功扱いしない。


In [ ]:
import zipfile
from google.colab import files
archive = pathlib.Path('/content') / f'vector-run-{MODEL}-results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zipped:
    for path in sorted((root / 'results' / MODEL).glob('*.json')):
        zipped.write(path, arcname=path.name)
files.download(str(archive))
